# **Diffusion Models: Forward Noise & Reverse Denoise**

Diffusion models learn to generate data by reversing a gradual noising process. Starting from clean data, the **forward process** incrementally adds Gaussian noise until the signal is destroyed. A neural network then learns the **reverse process** — predicting and removing noise step by step, eventually recovering structured samples from pure noise.

In this notebook, we train a DDPM ([Ho et al., "Denoising Diffusion Probabilistic Models" (NeurIPS 2020)](https://arxiv.org/abs/2006.11239)) on 2D point distribution and serialize the learned weights into GLSL for real-time rendering in WebGL.

## Steps:
1. Import necessary libraries and set up device
2. Define 2D point distribution dataset
3. Implement forward diffusion process
4. Define denoising network (MLP with positional encoding)
5. Train the model
6. Implement reverse diffusion
7. Serialize model weights for WebGL rendering

## Step 0: Imports and Setup

In [ ]:
!pip install opencv-python
!pip install Pillow
!pip install matplotlib
!pip install tqdm

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

DEVICE = torch.device("cpu")
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")

print(f"Using device: {DEVICE}")
torch.manual_seed(4242)
np.random.seed(4242)

## Step 1: 2D Point Cloud Dataset

In this assignment, we work with **2D point distributions**. Each dataset defines a continuous probability distribution $p(\mathbf{x})$ over $\mathbb{R}^2$ with geometric support: a circle is supported on a 1D curve, a checkerboard on a union of 2D regions, and an image-based dataset on the dark regions of the image. We draw $N$ samples from $p(\mathbf{x})$ to form the training set.

The goal of the generative model is to learn $p(\mathbf{x})$ from these samples and produce new ones.

We use simple 2D distributions for fast training and sampling here. But the same DDPM framework applies directly to much higher-dimensional settings: natural images, audio, and 3D shapes all define distributions over $\mathbb{R}^D$ with $D$ potentially in the hundreds of thousands (e.g., a 256×256 RGB image lives in $\mathbb{R}^{256\times256\times3}$). The key ideas of forward noising, learned denoising, and iterative sampling remain identical.

In [ ]:
def sample_circle(
    n_points: int, radius: float = 1.2, noise: float = 0.01
) -> torch.Tensor:
    """Sample n_points uniformly on a circle, with small Gaussian noise. Returns (n_points, 2)."""
    # sample uniform angles in [0, 2π]
    angles = torch.rand(n_points) * 2 * torch.pi
    x = radius * torch.cos(angles) + noise * torch.randn(n_points)
    y = radius * torch.sin(angles) + noise * torch.randn(n_points)
    return torch.stack([x, y], dim=1)


def sample_checkerboard(
    n_points: int, grid: int = 4, noise: float = 0.01
) -> torch.Tensor:
    """Sample n_points uniformly from the 'white' squares of a checkerboard in [-1, 1]^2. Returns (n_points, 2)."""
    cell = 2.0 / grid
    # enumerate bottom-left corners of all white squares
    ij = torch.tensor(
        [(i, j) for i in range(grid) for j in range(grid) if (i + j) % 2 == 0],
        dtype=torch.float32,
    )
    # randomly assign each point to a white square, then sample uniformly within it
    corners = ij[torch.randint(len(ij), (n_points,))]
    xy = -1.0 + (corners + torch.rand(n_points, 2)) * cell
    xy += noise * torch.randn_like(xy)
    return xy


def sample_image(
    n_points: int,
    path: str = "bee.png",
    mode: str = "binary",
    threshold: float = 0.1,
    noise: float = 0.01,
) -> torch.Tensor:
    """
    Sample n_points from an image, treating dark pixels as the support of the distribution.
    mode='binary': uniform over pixels darker than threshold.
    mode='density': weighted by pixel darkness.
    Returns (n_points, 2) with coordinates in [-1.5, 1.5]^2.
    """
    from PIL import Image

    # composite transparent background to white, then convert to grayscale
    raw = Image.open(path).convert("RGBA")
    bg = Image.new("RGBA", raw.size, (255, 255, 255, 255))
    bg.paste(raw, mask=raw.split()[3])
    img = np.array(bg.convert("L"), dtype=np.float32)
    H, W = img.shape

    if mode == "binary":
        rows, cols = np.where(img < threshold * 255)
        idx = np.random.choice(len(rows), size=n_points, replace=True)
        r, c = rows[idx], cols[idx]
    elif mode == "density":
        weights = (255.0 - img).ravel()
        weights = weights / weights.sum()
        flat_idx = np.random.choice(H * W, size=n_points, replace=True, p=weights)
        r, c = np.unravel_index(flat_idx, (H, W))
    else:
        raise ValueError(f"mode must be 'binary' or 'density', got '{mode}'")

    # map pixel coordinates to [-1.5, 1.5]^2, flipping y to match screen convention
    x = (c + 0.5) / W * 3.0 - 1.5
    y = -(r + 0.5) / H * 3.0 + 1.5
    pts = np.stack([x, y], axis=1).astype(np.float32)
    pts += noise * np.random.randn(*pts.shape)
    return torch.tensor(pts)


SAMPLERS = {
    "circle": sample_circle,
    "checkerboard": sample_checkerboard,
    "image": sample_image,
}


def get_data(name: str, n_points: int, **kwargs) -> torch.Tensor:
    assert name in SAMPLERS, f"Unknown dataset '{name}'. Choose from {list(SAMPLERS)}"
    return SAMPLERS[name](n_points, **kwargs)


# Choose your dataset here
# DATASET_NAME = 'circle'
# DATASET_NAME = 'checkerboard'
DATASET_NAME = "image"
data = get_data(DATASET_NAME, n_points=4000)


fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(data[:, 0].numpy(), data[:, 1].numpy(), s=4, alpha=0.5, color="steelblue")
ax.set_aspect("equal")
ax.set_xlim(-1.8, 1.8)
ax.set_ylim(-1.8, 1.8)
plt.tight_layout()
plt.show()

## Step 2: Forward Diffusion Process

The forward process is a **Markov chain** that gradually adds Gaussian noise to the data over $T$ steps:

$$q(\mathbf{x}_{1:T} \mid \mathbf{x}_0) = \prod_{t=1}^{T} q(\mathbf{x}_t \mid \mathbf{x}_{t-1}).$$

Each step adds a small amount of noise controlled by a variance schedule $\beta_1, \dots, \beta_T$:

$$q(\mathbf{x}_t \mid \mathbf{x}_{t-1}) = \mathcal{N}\!\left(\mathbf{x}_t;\; \sqrt{1 - \beta_t}\,\mathbf{x}_{t-1},\; \beta_t \mathbf{I}\right).$$

Define $\alpha_t = 1 - \beta_t$ and $\bar{\alpha}_t = \prod_{s=1}^{t} \alpha_s$. By the properties of Gaussians, we can directly sample $\mathbf{x}_t$ from $\mathbf{x}_0$ in closed form, skipping all intermediate steps:

$$q(\mathbf{x}_t \mid \mathbf{x}_0) = \mathcal{N}\!\left(\mathbf{x}_t;\; \sqrt{\bar{\alpha}_t}\,\mathbf{x}_0,\; (1 - \bar{\alpha}_t)\mathbf{I}\right).$$

Via reparameterization, sampling becomes:

$$\mathbf{x}_t = \sqrt{\bar{\alpha}_t}\,\mathbf{x}_0 + \sqrt{1 - \bar{\alpha}_t}\,\boldsymbol{\epsilon}, \qquad \boldsymbol{\epsilon} \sim \mathcal{N}(\mathbf{0}, \mathbf{I}).$$

As $t \to T$, $\bar{\alpha}_t \to 0$ and $\mathbf{x}_t$ converges to pure Gaussian noise. We use a linear schedule for $\beta_t$ and index it with a continuous $t \in [0, 1]$, mapped to integer indices internally.

**Your task.**
- The schedule quantities $\bar{\alpha}_t$ and other related quantities should be precomputed in `__init__`.
- Implement `q_sample` in `DiffusionSchedule`. Given a batch of clean points $\mathbf{x}_0$ and time steps $t$, use the closed-form reparameterization above to compute and return the noisy sample $\mathbf{x}_t$ and the noise $\boldsymbol{\epsilon}$. Use `_t_to_idx` to convert continuous $t \in [0, 1]$ to integer indices.


In [ ]:
class DiffusionSchedule:
    """
    Deffuion process with T timesteps, where noise level at step t is determined by beta_t.
    We use a linear noise schedule: beta_t = beta_start + t/T * (beta_end - beta_start).
    """

    def __init__(
        self,
        T: int = 400,
        beta_start: float = 1e-5,
        beta_end: float = 2e-3,
        device=DEVICE,
    ):
        self.T = T
        self.device = device
        self.betas = torch.linspace(beta_start, beta_end, T, device=device)

        # Compute any other quantities you need for the diffusion process here, and store them as attributes of this class.
        # Your implementation starts
        
        # Your implementation ends

    def _t_to_idx(self, t: torch.Tensor) -> torch.Tensor:
        """
        Convert time steps t (in [0, 1]) to indices.
        """
        return (t[:, 0] * self.T).long().clamp(0, self.T - 1)

    def q_sample(self, x0: torch.Tensor, t: torch.Tensor):
        """
        Forward noising: x_t = sqrt(alpha_bar_t)*x0 + sqrt(1-alpha_bar_t)*eps.
        Use Pytorch operations here to ensure compatibility with autograd.
        Args:
            x0: (B, 2)
            t:  (B, 1), float in [0, 1]
        Returns: xt (B, 2), epsilon (B, 2)
        """
        xt = torch.zeros_like(x0)
        epsilon = torch.randn_like(x0)

        # Your implementation starts

        # Your implementation ends
        return xt, epsilon


diffusion = DiffusionSchedule(device=DEVICE)


In [ ]:
# Visualize forward process. t expressed as float in [0, 1]
vis_data = data.to(DEVICE)[:2000]
t_vals_float = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]

fig, axes = plt.subplots(1, len(t_vals_float), figsize=(20, 3))
for ax, t_val in zip(axes, t_vals_float):
    t_tensor = torch.full((len(vis_data), 1), t_val, device=DEVICE)
    xt, _ = diffusion.q_sample(vis_data, t_tensor)
    pts = xt.cpu().numpy()
    ax.scatter(pts[:, 0], pts[:, 1], s=3, alpha=0.5, color='steelblue')
    ax.set_title(f't = {t_val:.2f}')
    ax.set_aspect('equal')
    ax.set_xlim(-3, 3)
    ax.set_ylim(-3, 3)

plt.tight_layout()
plt.show()


In [ ]:
# Create a video of the forward process, with t going from 0 to 1.
import cv2

FPS = 24
N_FRAMES_FWD = 200
VIS_N = 2000
W, H = 800, 800

vis_pts = data[:VIS_N].to(DEVICE)
t_vals_anim = torch.linspace(0.0, 1.0, N_FRAMES_FWD)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
vw = cv2.VideoWriter("forward.mp4", fourcc, FPS, (W, H))

for frame in range(N_FRAMES_FWD):
    t_val = t_vals_anim[frame].item()
    t_tensor = torch.full((VIS_N, 1), t_val, device=DEVICE)
    xt, _ = diffusion.q_sample(vis_pts, t_tensor)
    pts = xt.cpu().numpy()

    fig, ax = plt.subplots(figsize=(4, 4), dpi=100)
    ax.scatter(pts[:, 0], pts[:, 1], s=4, alpha=0.5, color="steelblue")
    ax.set_xlim(-3, 3)
    ax.set_ylim(-3, 3)
    ax.set_aspect("equal")
    ax.set_title(f"Forward Process t = {t_val:.2f}")
    fig.tight_layout()
    fig.canvas.draw()
    img = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8)
    img = img.reshape(fig.canvas.get_width_height()[::-1] + (4,))[:, :, :3]
    img = cv2.resize(img, (W, H))
    vw.write(cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
    plt.close(fig)

vw.release()
print("Saved forward.mp4")


## Step 3: Denoising Network

We train a neural network $\boldsymbol{\epsilon}_\theta(\mathbf{x}_t, t)$ to predict the noise $\boldsymbol{\epsilon}$ added to $\mathbf{x}_0$ at time step $t$. The network must be **time-conditioned**: the difficulty of the denoising task varies with $t$ — at small $t$ the input is nearly clean and only a subtle correction is needed, while at large $t$ the input is close to pure noise.

**Positional encoding.** A plain 2D coordinate is a weak input for an MLP. Borrowing from NeRF, we lift $\mathbf{x}_t$ with sinusoidal positional encoding to expose high-frequency spatial structure:

$$\gamma(\mathbf{x}_t) = \bigl[\mathbf{x}_t,\; \sin(2^0 \mathbf{x}_t),\; \cos(2^0 \mathbf{x}_t),\; \dots,\; \sin(2^{L-1} \mathbf{x}_t),\; \cos(2^{L-1} \mathbf{x}_t)\bigr],$$

where $L$ is the number of frequency bands. The time scalar $t \in [0, 1]$ is concatenated directly without encoding.

**Architecture.** The encoded input is passed through a small MLP with 3 hidden layers of width 48 and ReLU activations, followed by a linear output layer that produces a 2D predicted noise vector.

**Your task.**
- Copy your positional encoding implementation from the A3b NeRF assignment into `_positional_encoding`.
- Read through `forward` to make sure you understand how the encoded input is assembled and passed through the network.

In [ ]:
class NoisePredictorMLP(nn.Module):
    """
    MLP that predicts noise epsilon from (x_t, t).
    x_t is lifted with NeRF-style positional encoding; t is passed as a raw scalar.
    """
    def __init__(self, hidden_dim: int = 48, n_layers: int = 3, n_freqs: int = 8):
        super().__init__()
        self.n_freqs = n_freqs
        xt_dim = 2 * (1 + 2 * n_freqs)
        # (Coordinates + time)
        in_dim = xt_dim + 1  
        layers = [nn.Linear(in_dim, hidden_dim), nn.ReLU()]
        for _ in range(n_layers - 1):
            layers += [nn.Linear(hidden_dim, hidden_dim), nn.ReLU()]
        layers.append(nn.Linear(hidden_dim, 2))
        self.net = nn.Sequential(*layers)

    def _positional_encoding(self, x: torch.Tensor) -> torch.Tensor:
        """
        Apply positional encoding to x.
        Input shape (B, 2), output shape (B, 2*(1+2*n_freqs)).
        """
        # You can borrow code from your NeRF implementation
        # Your implementation starts
        encodings = [x]
        # Your implementation ends
        return torch.cat(encodings, dim=-1)

    def forward(self, xt: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        xt_enc = self._positional_encoding(xt)
        h = torch.cat([xt_enc, t], dim=-1)
        return self.net(h)


# Check the shape can pass through the model
model = NoisePredictorMLP().to(DEVICE)
dummy_xt = torch.randn(16, 2, device=DEVICE)
dummy_t  = torch.rand(16, 1, device=DEVICE)
out = model(dummy_xt, dummy_t)
print(f'Output shape: {out.shape}')
print(f'Parameters:   {sum(p.numel() for p in model.parameters()):,}')


## Step 4: Training

The model is trained by minimizing the mean squared error between the predicted noise and the true noise. For a clean sample $\mathbf{x}_0$, a random time step $t$, and noise $\boldsymbol{\epsilon} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$, the loss is:

$$\mathcal{L} = \mathbb{E}_{t,\, \mathbf{x}_0,\, \boldsymbol{\epsilon}}\bigl[\| \boldsymbol{\epsilon} - \boldsymbol{\epsilon}_\theta(\mathbf{x}_t, t) \|^2 \bigr].$$

At each training iteration, we sample a random batch of data points and a random $t$ for each point, corrupt the data with the forward process, and supervise the network to recover the noise.

**Your task.**
- Implement `train_step`. Each call should: (1) sample $t$ uniformly from $[0, 1]$; (2) compute $\mathbf{x}_t$ using `q_sample`; (3) predict the noise with the network; (4) compute (and return) the MSE loss; (5) perform a gradient update.

**Note on training.** The model requires a large number of epochs to converge. You may find that the loss stops decreasing after a short number of epochs — this is expected. Think about why the training loss may not be a good metric for evaluating generation quality.

In [ ]:
def train_step(model, schedule, x0_batch, optimizer, scheduler) -> float:
    """
    Perform one training step. Returns the loss value as a float.
    """
    model.train()
    B = x0_batch.shape[0]
    loss = 0.0

    # Steps:
    # 1. Sample t uniformly in [0, 1]
    # 2. Forward parameters (x0, t) to q_sample to get (xt, epsilon)
    # 3. Predict noise with model(xt, t)
    # 4. Compute MSE loss between predicted noise and true noise

    # Your implementation starts
    
    # Your implementation ends

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()
    return loss.item()

In [ ]:
N_TRAIN = 8000
BATCH_SIZE = 512
N_EPOCHS = 40000
LR = 1e-3
ETA_MIN = 1e-5

train_data = get_data(DATASET_NAME, n_points=N_TRAIN).to(DEVICE)
model = NoisePredictorMLP().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=N_EPOCHS, eta_min=ETA_MIN
)
diffusion = DiffusionSchedule(device=DEVICE)

loss_history = []
pbar = tqdm(range(N_EPOCHS), desc="Training")
for epoch in pbar:
    idx = torch.randint(0, N_TRAIN, (BATCH_SIZE,))
    x0_batch = train_data[idx]
    loss = train_step(model, diffusion, x0_batch, optimizer, scheduler)
    loss_history.append(loss)
    if (epoch + 1) % 500 == 0:
        pbar.set_postfix({"loss": f"{loss:.4f}"})

plt.figure(figsize=(8, 3))
plt.plot(loss_history, linewidth=0.8)
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Training Loss")
plt.tight_layout()
plt.show()


## Step 5: Reverse Diffusion — Sampling

The reverse process is also a Markov chain, but runs backwards in time. The learned reverse transition is defined as:

$$p_\theta(\mathbf{x}_{t-1} \mid \mathbf{x}_t) = \mathcal{N}\!\left(\mathbf{x}_{t-1};\; \boldsymbol{\mu}_\theta(\mathbf{x}_t, t),\; \boldsymbol{\Sigma}_\theta(\mathbf{x}_t, t)\right).$$

where the mean $\boldsymbol{\mu}_\theta$ is parameterized via the noise predictor $\boldsymbol{\epsilon}_\theta$:

$$\boldsymbol{\mu}_\theta(\mathbf{x}_t, t) = \frac{1}{\sqrt{\alpha_t}} \Bigl(\mathbf{x}_t - \frac{1 - \alpha_t}{\sqrt{1 - \bar{\alpha}_t}}\, \boldsymbol{\epsilon}_\theta(\mathbf{x}_t, t)\Bigr).$$

In general, $\boldsymbol{\Sigma}_\theta$ could itself be a learned matrix output by the network. Ho et al. found that fixing it to a scalar multiple of the identity — $\boldsymbol{\Sigma}_\theta = \sigma_t^2 \mathbf{I}$ — works just as well in practice. Here we adopt $\sigma_t^2 = \tilde{\beta}_t$ as that scalar:

$$\tilde{\beta}_t = \frac{1 - \bar{\alpha}_{t-1}}{1 - \bar{\alpha}_t}\,\beta_t.$$

Combining the mean and the fixed variance, each reverse step becomes:

$$\mathbf{x}_{t-1} = \frac{1}{\sqrt{\alpha_t}} \Bigl(\mathbf{x}_t - \frac{1 - \alpha_t}{\sqrt{1 - \bar{\alpha}_t}}\, \boldsymbol{\epsilon}_\theta(\mathbf{x}_t, t)\Bigr) + \sqrt{\tilde{\beta}_t}\, \mathbf{z}, \qquad \mathbf{z} \sim \mathcal{N}(\mathbf{0}, \mathbf{I}).$$

At the final step $t = 0$, no noise is added ($\mathbf{z} = \mathbf{0}$). Starting from $\mathbf{x}_T \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$ and iterating this step for $t = T-1, \dots, 0$ produces a sample from the learned distribution.

**Your task.**
- Before implementing `ddpm_sample`, go back to `DiffusionSchedule.__init__()` and make sure $\alpha_t$, $\bar{\alpha}_t$, and $\tilde{\beta}_t$ (and other quantities) are precomputed and stored as attributes. The sampler will look them up at every step.
- Implement the core reverse step inside `ddpm_sample`. At each step, query the network for $\boldsymbol{\epsilon}_\theta(\mathbf{x}_t, t)$, compute the denoised mean $\boldsymbol{\mu}_\theta$, and sample $\mathbf{x}_{t-1}$ using the equation above.
- Run the sampler and visualize the generated samples. Do they match the training distribution?

In [ ]:
@torch.no_grad()
def ddpm_sample(model, diffusion, n_samples: int = 4000, return_trajectory: int = 0):
    """
    Run the DDPM reverse diffusion process to generate samples.
    """
    model.eval()
    T = diffusion.T

    # Sample initial noise: x_T ~ N(0, I)
    xt = torch.randn(n_samples, 2, device=diffusion.device)

    # Decide which steps to snapshot for trajectory visualization
    if return_trajectory:
        snap_steps = (
            set(
                int(T - 1 - i * (T - 1) / (return_trajectory - 1))
                for i in range(return_trajectory)
            )
            if return_trajectory > 1
            else {T - 1}
        )
        trajectory = []

    # Reverse loop: step from T-1 down to 0
    for step in reversed(range(T)):
        # Steps:
        # 1. Convert integer step to continuous t in [0, 1] for the network
        # 2. Retrieve alpha_t, alpha_bar_t, tilde_beta_t from the schedule at this step
        # 3. Predict noise: eps_pred = model(xt, t_float)
        # 4. Compute the reverse mean mu_theta using the formula from Step 5
        # 5. Sample x_{t-1} by adding noise scaled by sqrt(tilde_beta_t);
        #    skip noise injection at step 0.

        # Your implementation starts

        # Your implementation ends

        if return_trajectory and step in snap_steps:
            trajectory.append((xt.cpu(), step / T))

    if return_trajectory:
        trajectory = list(reversed(trajectory))
        return xt, trajectory
    return xt


In [ ]:
# Generate samples from the trained model, and compare with ground truth.
generated, trajectory = ddpm_sample(
    model, diffusion, n_samples=4000, return_trajectory=6
)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
axes[0].scatter(
    train_data[:4000, 0].cpu(),
    train_data[:4000, 1].cpu(),
    s=4,
    alpha=0.5,
    color="steelblue",
)
axes[0].set_title("Ground Truth")
axes[0].set_aspect("equal")
axes[0].set_xlim(-1.8, 1.8)
axes[0].set_ylim(-1.8, 1.8)

axes[1].scatter(
    generated[:, 0].cpu(), generated[:, 1].cpu(), s=4, alpha=0.5, color="coral"
)
axes[1].set_title("DDPM Generated")
axes[1].set_aspect("equal")
axes[1].set_xlim(-1.8, 1.8)
axes[1].set_ylim(-1.8, 1.8)

plt.tight_layout()
plt.show()


In [ ]:
# Visualize the reverse diffusion trajectory, showing how the noise gradually transforms into data.
fig, axes = plt.subplots(1, len(trajectory), figsize=(20, 3))
for ax, (pts, t_val) in zip(axes, trajectory):
    ax.scatter(pts[:, 0], pts[:, 1], s=3, alpha=0.5, color='coral')
    ax.set_title(f't = {t_val:.2f}')
    ax.set_aspect('equal')
    ax.set_xlim(-3, 3)
    ax.set_ylim(-3, 3)

plt.tight_layout()
plt.show()


In [ ]:
# Create a video of the reverse diffusion process, with t going from 1 to 0.
import cv2

FPS  = 20
W, H = 800, 800

_, rev_traj = ddpm_sample(model, diffusion, n_samples=2000, return_trajectory=200)

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
vw = cv2.VideoWriter('reverse.mp4', fourcc, FPS, (W, H))

for pts, t_val in reversed(rev_traj):
    fig, ax = plt.subplots(figsize=(4, 4), dpi=100)
    ax.scatter(pts[:, 0], pts[:, 1], s=4, alpha=0.5, color='coral')
    ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
    ax.set_aspect('equal')
    ax.set_title(f'Reverse Process t = {t_val:.2f}')
    fig.tight_layout()
    fig.canvas.draw()
    img = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8)
    img = img.reshape(fig.canvas.get_width_height()[::-1] + (4,))[:, :, :3]
    img = cv2.resize(img, (W, H))
    vw.write(cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
    plt.close(fig)

vw.release()
print('Saved reverse.mp4')


## Step 6: Serialize Model

Following the same approach as the NeRF assignment, we serialize the trained network weights into a GLSL function that can be embedded directly in the WebGL shader.


**What to do.** Run the cell below. It will write the full GLSL function to `serialized_model.txt`. Open that file and paste its contents into the corresponding section of `fragment.glsl`.

In [ ]:
# Serialize Model
import re


def fmt_float(x):
    s = f"{x:.4f}"
    s = re.sub(r"\b0\.", ".", s)
    return s


def print_vec4(v):
    return "vec4(" + ",".join(fmt_float(float(x)) for x in v) + ")"


def print_mat4(m):
    """m is [4, 4] numpy, rows=output neurons, cols=input neurons."""
    mT = m.T
    vals = []
    for r in range(4):
        for c in range(4):
            vals.append(fmt_float(float(mT[r, c])))
    return "mat4(" + ",".join(vals) + ")"


def generate_pe_glsl(n_freqs):
    """
    Pack the positional encoding of xt (2D) plus scalar t into vec4 chunks.
    """
    lines = []
    lines.append("    vec4 PE_0 = vec4(xt.x,xt.y,sin(1.00*xt.x),sin(1.00*xt.y));")
    for j in range(1, n_freqs + 1):
        fp = 2.0 ** (j - 1)
        fc = 2.0**j
        fp_s = f"{fp:.2f}"
        fc_s = f"{fc:.2f}"
        if j < n_freqs:
            lines.append(
                f"    vec4 PE_{j} = vec4(cos({fp_s}*xt.x),cos({fp_s}*xt.y),sin({fc_s}*xt.x),sin({fc_s}*xt.y));"
            )
        else:
            lines.append(
                f"    vec4 PE_{j} = vec4(cos({fp_s}*xt.x),cos({fp_s}*xt.y),t,0.0);"
            )
    return lines


def serialize_linear(
    w, b, in_prefix, out_prefix, n_in_chunks, n_out_chunks, use_relu=True
):
    """Emit GLSL for one fully-connected layer. w: [out_dim, in_dim];  b: [out_dim]"""
    pad = (4 - w.shape[1] % 4) % 4
    if pad:
        w = np.pad(w, [(0, 0), (0, pad)])

    lines = []
    for i in range(n_out_chunks):
        r0, r1 = i * 4, min((i + 1) * 4, w.shape[0])
        w_sub = w[r0:r1, :]
        b_sub = b[r0:r1]
        if w_sub.shape[0] < 4:
            w_sub = np.pad(w_sub, [(0, 4 - w_sub.shape[0]), (0, 0)])
            b_sub = np.pad(b_sub, [(0, 4 - b_sub.shape[0])])

        terms = [
            f"{print_mat4(w_sub[:, j * 4 : (j + 1) * 4])} * {in_prefix}_{j}"
            for j in range(n_in_chunks)
        ]
        terms.append(print_vec4(b_sub))

        sep = " +\n    "
        if use_relu:
            lines.append(f"    vec4 {out_prefix}_{i} = relu(\n    {sep.join(terms)});")
        else:
            lines.append(f"    vec4 {out_prefix}_{i} = \n    {sep.join(terms)};")
    return lines


def serialize_output(w, b, in_prefix, n_in_chunks, out_dim):
    """Emit GLSL for the final linear layer using dot products (no activation)."""
    pad = (4 - w.shape[1] % 4) % 4
    if pad:
        w = np.pad(w, [(0, 0), (0, pad)])

    lines = []
    for o in range(out_dim):
        wrow = w[o]
        terms = [
            f"dot({in_prefix}_{j},{print_vec4(wrow[j * 4 : (j + 1) * 4])})"
            for j in range(n_in_chunks)
        ]
        lines.append(
            f"    float out{o} = " + "+".join(terms) + f"+{fmt_float(float(b[o]))};"
        )
    return lines


def serialize_ddpm_model(model):
    """
    Serialize NoisePredictorMLP to a GLSL vec2 queryNetwork(vec2 xt, float t) function.
    Automatically adapts to any hidden_dim, n_layers, and n_freqs stored on the model.
    """
    n_freqs = model.n_freqs
    linears = [m for m in model.net if isinstance(m, nn.Linear)]
    hidden_dim = linears[0].out_features
    n_pe_chunks = n_freqs + 1
    n_hid = (hidden_dim + 3) // 4
    out_dim = linears[-1].out_features

    lines = ["vec2 queryNetwork(vec2 xt, float t){"]
    lines += generate_pe_glsl(n_freqs)
    lines.append("")

    # All layers except the last (hidden layers with ReLU)
    in_prefix, n_in = "PE", n_pe_chunks
    for idx, lin in enumerate(linears[:-1]):
        w = lin.weight.detach().cpu().numpy()
        b = lin.bias.detach().cpu().numpy()
        out_prefix = f"f{idx}"
        lines += serialize_linear(w, b, in_prefix, out_prefix, n_in, n_hid)
        lines.append("")
        in_prefix, n_in = out_prefix, n_hid

    # Output layer (no activation)
    w_out = linears[-1].weight.detach().cpu().numpy()
    b_out = linears[-1].bias.detach().cpu().numpy()
    lines += serialize_output(w_out, b_out, f"f{len(linears) - 2}", n_hid, out_dim)
    lines.append("")
    lines.append(
        "    return vec2(" + ",".join(f"out{o}" for o in range(out_dim)) + ");"
    )
    lines.append("}")

    return "\n".join(lines)


glsl_code = serialize_ddpm_model(model)

filename = "serialized_model.txt"
with open(filename, "w", encoding="utf-8") as f:
    f.write(glsl_code)

n_linears = sum(1 for m in model.net if isinstance(m, nn.Linear))
print(
    f"Model: n_freqs={model.n_freqs}, hidden_dim={list(model.net.children())[0].out_features}, "
    f"n_hidden_layers={n_linears - 1}, out_dim={list(model.net.children())[-1].out_features}"
)
print(f"Saved {len(glsl_code):,} chars to {filename}")
print("\n--- Preview (first 800 chars) ---")
print(glsl_code[:800])
